# Project 4: Real-Time Event Streaming Analytics with Watermarking and Checkpoints

**Module**: 13_end_to_end_projects  
**Notebook**: 04_streaming_analytics_pipeline.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Complete end-to-end real-time streaming pipeline: simulating incoming JSON event streams, tumbling window aggregations, watermarking, checkpointing, and failure recovery.
- **Why does this topic exist?**: Real-time stream processing is a cornerstone of modern event-driven architectures.
- **What problem does it solve?**: Handling streaming event ingestion without external Kafka dependencies in local Docker clusters.
- **Where is it used in real-world Data Engineering?**: Live operational telemetry and real-time revenue monitoring dashboards.

---

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Streaming_Analytics_Project")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Setting Up Directory Stream Paths and Synthetic Event Generator

In [ ]:
import json

stream_input = "/opt/spark-data/streaming/events"
chk_path = "/opt/spark-data/warehouse/project_stream_chk"
out_path = "/opt/spark-data/warehouse/project_stream_out"

# Reset directories
os.system(f"rm -rf {stream_input}/* {chk_path} {out_path}")
os.makedirs(stream_input, exist_ok=True)

event_schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("order_ts", StringType()),
    StructField("channel", StringType()),
    StructField("revenue", DoubleType())
])

# Read streaming files
events_stream = (
    spark.readStream
    .schema(event_schema)
    .json(stream_input)
    .withColumn("timestamp", F.to_timestamp("order_ts"))
    .withWatermark("timestamp", "10 seconds")
)

print("Streaming input table configured. isStreaming:", events_stream.isStreaming)

## 4. Real-Time Windowed Revenue Aggregations

In [ ]:
# Windowed revenue by channel over 5-second tumbling windows
windowed_rev = (
    events_stream
    .groupBy(
        F.window("timestamp", "5 seconds"),
        F.col("channel")
    )
    .agg(
        F.count("order_id").alias("order_count"),
        F.round(F.sum("revenue"), 2).alias("total_channel_revenue")
    )
)

# Start streaming query writing to memory sink
query = (
    windowed_rev
    .writeStream
    .format("memory")
    .queryName("live_sales_dashboard")
    .outputMode("complete")
    .start()
)

print(f"Streaming query started with Query ID: {query.id}")

---

## Experiment: Simulating Micro-Batch Event Ingestion and Live Querying

- **1. What are we testing?**: Drop event files into stream_input and query the live_sales_dashboard table.
- **2. What do we expect?**: Dashboard updates in real time as files are written.

In [ ]:
# Simulate 2 event bursts
for batch_idx in range(1, 3):
    events = [
        {"order_id": 100 + batch_idx*10 + i, "order_ts": "2026-11-09 20:00:00", "channel": "Mobile" if i%2==0 else "Web", "revenue": 100.0 * i}
        for i in range(1, 4)
    ]
    with open(f"{stream_input}/batch_{batch_idx}.json", "w") as f:
        for ev in events:
            f.write(json.dumps(ev) + "\n")
    time.sleep(2)

query.processAllAvailable()

print("=== LIVE STREAMING SALES DASHBOARD ===")
spark.sql("SELECT window.start, window.end, channel, order_count, total_channel_revenue FROM live_sales_dashboard ORDER BY total_channel_revenue DESC").show(truncate=False)

query.stop()
print("Streaming pipeline completed and stopped.")

**3. What happened?**: Live streaming aggregation updated dynamically as new files landed.  
**4. Why did it happen?**: Micro-batch query recomputed the windowed result table in Complete mode.  
**5. Practical Engineering Lesson**: Structured Streaming provides unified DataFrame syntax for both real-time event streams and batch files.

---

## Practical Exercises

### Exercise 1 (Beginner): Check Query Last Progress
**Problem Statement**: Inspect query.lastProgress dictionary.  
**Input / Context**: query

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Parquet Streaming Sink
**Problem Statement**: Write a stream in Append mode to Parquet sink with checkpoint.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Stateful Fault Recovery
**Problem Statement**: Explain how restarting the query with the same checkpoint resumes state.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check Query Last Progress):
```python
print('query.lastProgress provides inputRowsPerSecond, processedRowsPerSecond, batchId.')
```

#### Solution 2 (Parquet Streaming Sink):
```python
print('events_stream.writeStream.format("parquet").option("checkpointLocation", chk).start(out)')
```

#### Solution 3 (Stateful Fault Recovery):
```python
print('Structured Streaming loads state store snapshots from state/ and offset WAL from offsets/.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Unified Paradigm**: Batch and Streaming share the exact same DataFrame / SQL APIs.
- **Watermarking & Checkpoints**: Mandatory for state bounding and exactly-once recovery.

### Essential Technical Terms
- `Tumbling Window`: Fixed-duration non-overlapping time window.
- `Complete Output Mode`: Emits full result table on every trigger.

### Common Engineering Mistakes
- **Omitting Checkpoint in Production**: Stream cannot recover from crash without checkpoints.

### Interview Questions on This Topic
- **Q: What happens if a streaming query crashes and is restarted with the same checkpointLocation?**
  - *A*: It reads the offset log and state store from the checkpoint directory, resuming exactly where it left off with zero data loss or duplicate processing.

### What to Study Next
Curriculum complete! Review documentation in docs/ and project README.md.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')